# Clustering and behavior-profile analysis

K-means clustering with numerical safety, efficiency, and comfort features plus the categorical dominant critical position.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA


# ============================================================
# Settings
# ============================================================
summary_path = "results/summary/exid_metrics_summary_all_recordings_cleaned.csv"

# For testing only, you can use:
# summary_path = "exid_metrics_summary_recording_02.csv"

random_state = 42

k_min = 2
k_max = 8

# If you want to manually force k, set this to an integer, e.g. 4.
# If None, the code selects the k with the highest silhouette score.
manual_k = None


# ============================================================
# Load data
# ============================================================
df = pd.read_csv(summary_path)

print("Loaded shape:", df.shape)
display(df.head())


# ============================================================
# Keep only successful rows
# ============================================================
if "status" in df.columns:
    df = df[df["status"] == "ok"].copy()

print("Valid rows:", len(df))

if len(df) < 20:
    print(
        "Warning: this dataset is very small for clustering. "
        "Use the full all-recordings summary for meaningful results."
    )


# ============================================================
# Feature selection
# ============================================================
numeric_features = [
    # Safety
    "max_pcad",
    "mean_pcad",

    # Efficiency
    "average_speed",
    "max_speed",

    # Comfort
    "max_longitudinal_jerk",
    "max_lateral_jerk",
    "rms_total_jerk",
]

categorical_features = [
    "dominant_critical_position"
]

# Keep only columns that actually exist
numeric_features = [col for col in numeric_features if col in df.columns]
categorical_features = [col for col in categorical_features if col in df.columns]

feature_cols = numeric_features + categorical_features

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal selected features before encoding:", len(feature_cols))
print(feature_cols)


# ============================================================
# Prepare feature matrix
# ============================================================
X = df[feature_cols].copy()

# Replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Make sure categorical column is string-like
for col in categorical_features:
    X[col] = X[col].astype("object")

print("\nRaw feature matrix shape:", X.shape)


# ============================================================
# Preprocessing
# ============================================================
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Compatibility with different sklearn versions
try:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]
    )
except TypeError:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ]
    )

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

X_processed = preprocess.fit_transform(X)

print("Processed feature matrix shape:", X_processed.shape)


# ============================================================
# Get processed feature names
# ============================================================
processed_feature_names = []

processed_feature_names.extend(numeric_features)

if len(categorical_features) > 0:
    onehot = preprocess.named_transformers_["cat"].named_steps["onehot"]
    onehot_feature_names = onehot.get_feature_names_out(categorical_features)
    processed_feature_names.extend(onehot_feature_names)

print("\nProcessed feature names:")
print(processed_feature_names)


# ============================================================
# Find a reasonable number of clusters
# ============================================================
scores = []

for k in range(k_min, k_max + 1):

    if len(df) <= k:
        continue

    kmeans = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=50
    )

    labels = kmeans.fit_predict(X_processed)

    sil = silhouette_score(X_processed, labels)
    db = davies_bouldin_score(X_processed, labels)
    ch = calinski_harabasz_score(X_processed, labels)

    scores.append(
        {
            "k": k,
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        }
    )

scores_df = pd.DataFrame(scores)

display(scores_df)


# ============================================================
# Plot clustering scores
# ============================================================
plt.figure(figsize=(8, 4))
plt.plot(scores_df["k"], scores_df["silhouette"], marker="o")
plt.xlabel("Number of clusters k")
plt.ylabel("Silhouette score")
plt.title("K-means silhouette score by k")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(scores_df["k"], scores_df["davies_bouldin"], marker="o")
plt.xlabel("Number of clusters k")
plt.ylabel("Davies-Bouldin score")
plt.title("K-means Davies-Bouldin score by k")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(scores_df["k"], scores_df["calinski_harabasz"], marker="o")
plt.xlabel("Number of clusters k")
plt.ylabel("Calinski-Harabasz score")
plt.title("K-means Calinski-Harabasz score by k")
plt.grid(True)
plt.show()


# ============================================================
# Choose final k
# ============================================================
if manual_k is None:
    best_k = int(
        scores_df
        .sort_values("silhouette", ascending=False)
        .iloc[0]["k"]
    )
    print("Selected best_k based on silhouette:", best_k)
else:
    best_k = int(manual_k)
    print("Selected manual_k:", best_k)


# ============================================================
# Fit final K-means model
# ============================================================
final_model = KMeans(
    n_clusters=best_k,
    random_state=random_state,
    n_init=100
)

df["cluster"] = final_model.fit_predict(X_processed)


# ============================================================
# PCA visualization
# ============================================================
pca = PCA(n_components=2, random_state=random_state)
X_pca = pca.fit_transform(X_processed)

df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]

print("PCA explained variance ratio:", pca.explained_variance_ratio_)


plt.figure(figsize=(8, 6))

for cluster_id in sorted(df["cluster"].unique()):
    cluster_df = df[df["cluster"] == cluster_id]

    plt.scatter(
        cluster_df["pca_1"],
        cluster_df["pca_2"],
        label=f"Cluster {cluster_id}",
        alpha=0.7
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-means clusters visualized by PCA")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# Cluster size
# ============================================================
cluster_size = (
    df["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="count")
)

cluster_size["percentage"] = (
    cluster_size["count"] / cluster_size["count"].sum() * 100
)

display(cluster_size)


# ============================================================
# Cluster profile: numerical features
# ============================================================
numeric_cluster_profile = (
    df.groupby("cluster")[numeric_features]
    .mean()
    .round(3)
)

display(numeric_cluster_profile)


# ============================================================
# Cluster profile: dominant critical position distribution
# ============================================================
if "dominant_critical_position" in df.columns:
    dominant_position_distribution = (
        pd.crosstab(
            df["cluster"],
            df["dominant_critical_position"],
            normalize="index"
        )
        .mul(100)
        .round(1)
    )

    display(dominant_position_distribution)


# ============================================================
# Optional: cluster profile with extra useful columns
# ============================================================
extra_profile_cols = [
    col for col in [
        "p95_pcad",
        "high_pcad_duration_ratio",
        "critical_position_switch_count",
        "freq_No_Risk_percent",
        "duration_s",
        "valid_pairwise_pcad_count",
        "positive_pairwise_pcad_count"
    ]
    if col in df.columns
]

if len(extra_profile_cols) > 0:
    extra_cluster_profile = (
        df.groupby("cluster")[extra_profile_cols]
        .mean()
        .round(3)
    )

    display(extra_cluster_profile)


# ============================================================
# Inspect cluster centers in processed feature space
# ============================================================
cluster_centers = pd.DataFrame(
    final_model.cluster_centers_,
    columns=processed_feature_names
)

cluster_centers.index.name = "cluster"

display(cluster_centers.round(3))


# ============================================================
# Save clustered result
# ============================================================
clustered_output_path = summary_path.replace(
    ".csv",
    "_selected_features_dominant_position_kmeans_clustered.csv"
)

df.to_csv(clustered_output_path, index=False)

print("Clustered result saved to:", clustered_output_path)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# ============================================================
# 3D PCA visualization
# ============================================================
pca_3d = PCA(n_components=3, random_state=random_state)
X_pca_3d = pca_3d.fit_transform(X_processed)

df["pca_1"] = X_pca_3d[:, 0]
df["pca_2"] = X_pca_3d[:, 1]
df["pca_3"] = X_pca_3d[:, 2]

print("3D PCA explained variance ratio:")
print(pca_3d.explained_variance_ratio_)

print("Total explained variance:")
print(pca_3d.explained_variance_ratio_.sum())

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for cluster_id in sorted(df["cluster"].unique()):
    cluster_df = df[df["cluster"] == cluster_id]

    ax.scatter(
        cluster_df["pca_1"],
        cluster_df["pca_2"],
        cluster_df["pca_3"],
        label=f"Cluster {cluster_id}",
        alpha=0.7
    )

ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.set_zlabel("PCA 3")
ax.set_title("Merging vehicle clusters visualized by 3D PCA")
ax.legend()

plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


def compute_dimension_pca_score(df, feature_cols, score_name):
    """
    Compute the first principal component for one evaluation dimension.
    """

    feature_cols = [col for col in feature_cols if col in df.columns]

    if len(feature_cols) == 0:
        raise ValueError(f"No valid features found for {score_name}.")

    X_dim = df[feature_cols].replace([np.inf, -np.inf], np.nan)

    pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=1))
        ]
    )

    score = pipe.fit_transform(X_dim).ravel()

    explained_var = pipe.named_steps["pca"].explained_variance_ratio_[0]

    print(f"{score_name} features:", feature_cols)
    print(f"{score_name} PC1 explained variance: {explained_var:.3f}")

    return score, pipe


# ============================================================
# Define three dimensions
# ============================================================
safety_features = [
    "max_pcad",
    "mean_pcad"
]

efficiency_features = [
    "average_speed",
    "max_speed"
]

comfort_features = [
    "max_longitudinal_jerk",
    "max_lateral_jerk",
    "rms_total_jerk"
]


# ============================================================
# Compute dimension-level PCA scores
# ============================================================
df["safety_pc1"], safety_pca_pipe = compute_dimension_pca_score(
    df,
    safety_features,
    "Safety"
)

df["efficiency_pc1"], efficiency_pca_pipe = compute_dimension_pca_score(
    df,
    efficiency_features,
    "Efficiency"
)

df["comfort_pc1"], comfort_pca_pipe = compute_dimension_pca_score(
    df,
    comfort_features,
    "Comfort"
)


# ============================================================
# 3D plot: Safety-Efficiency-Comfort space
# ============================================================
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for cluster_id in sorted(df["cluster"].unique()):
    cluster_df = df[df["cluster"] == cluster_id]

    ax.scatter(
        cluster_df["safety_pc1"],
        cluster_df["efficiency_pc1"],
        cluster_df["comfort_pc1"],
        label=f"Cluster {cluster_id}",
        alpha=0.7
    )

ax.set_xlabel("Safety PC1")
ax.set_ylabel("Efficiency PC1")
ax.set_zlabel("Comfort PC1")
ax.set_title("Clusters in safety-efficiency-comfort PCA space")
ax.legend()

plt.show()

In [ ]:
# ============================================================
# Violin plots for cluster interpretation
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional but recommended for easier violin plots
import seaborn as sns


# ============================================================
# Settings
# ============================================================
cluster_col = "cluster"

figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)


# ============================================================
# Metrics for violin plots
# ============================================================
violin_metrics = [
    # Safety
    "max_pcad",
    "mean_pcad",

    # Efficiency
    "average_speed",
    "max_speed",

    # Comfort
    "max_longitudinal_jerk",
    "max_lateral_jerk",
    "rms_total_jerk",
]

violin_metrics = [col for col in violin_metrics if col in df.columns]

print("Metrics used for violin plots:")
print(violin_metrics)


# ============================================================
# Clean plotting data
# ============================================================
plot_df = df[[cluster_col] + violin_metrics].copy()
plot_df = plot_df.replace([np.inf, -np.inf], np.nan)

# Make cluster labels ordered
plot_df[cluster_col] = plot_df[cluster_col].astype(int)
cluster_order = sorted(plot_df[cluster_col].dropna().unique())

print("Cluster order:", cluster_order)


# ============================================================
# Metric labels for figures
# ============================================================
metric_labels = {
    "max_pcad": "Maximum PCAD",
    "mean_pcad": "Mean PCAD",
    "average_speed": "Average Speed",
    "max_speed": "Maximum Speed",
    "max_longitudinal_jerk": "Maximum Longitudinal Jerk",
    "max_lateral_jerk": "Maximum Lateral Jerk",
    "rms_total_jerk": "RMS Total Jerk",
}


# ============================================================
# Draw violin plots in one figure
# ============================================================
n_metrics = len(violin_metrics)
n_cols = 2
n_rows = int(np.ceil(n_metrics / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(12, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for i, metric in enumerate(violin_metrics):
    ax = axes[i]

    sns.violinplot(
        data=plot_df,
        x=cluster_col,
        y=metric,
        order=cluster_order,
        inner="box",
        cut=0,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x=cluster_col,
        y=metric,
        order=cluster_order,
        size=3,
        alpha=0.45,
        jitter=True,
        ax=ax
    )

    ax.set_title(metric_labels.get(metric, metric))
    ax.set_xlabel("Cluster")
    ax.set_ylabel(metric_labels.get(metric, metric))
    ax.grid(True, axis="y", alpha=0.3)

# Remove empty subplot if needed
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

violin_output_path = os.path.join(figures_dir, "cluster_violin_plots_selected_metrics.png")
plt.savefig(violin_output_path, dpi=300, bbox_inches="tight")
plt.show()

print("Violin plot saved to:", violin_output_path)

In [ ]:
# ============================================================
# Dominant critical position distribution by cluster
# ============================================================
if "dominant_critical_position" in df.columns:

    dominant_dist = (
        pd.crosstab(
            df[cluster_col],
            df["dominant_critical_position"],
            normalize="index"
        )
        .mul(100)
        .round(1)
    )

    display(dominant_dist)

    ax = dominant_dist.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 5)
    )

    ax.set_xlabel("Cluster")
    ax.set_ylabel("Percentage (%)")
    ax.set_title("Dominant Critical Position Distribution by Cluster")
    ax.legend(
        title="Dominant Critical Position",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()

    dominant_output_path = os.path.join(
        figures_dir,
        "dominant_critical_position_distribution_by_cluster.png"
    )

    plt.savefig(dominant_output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Dominant critical position figure saved to:", dominant_output_path)